In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
)

print("Spark version:", spark.version)
print("Databricks RAG pipeline initialized")

Spark version: 4.2.0
Databricks RAG pipeline initialized


In [0]:
raw_df = spark.table("workspace.default.rag_corpus_raw")

print("Raw document count:", raw_df.count())
raw_df.printSchema()

display(raw_df)

Raw document count: 12
root
 |-- doc_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- text: string (nullable = true)



doc_id,title,category,text
D1,Introduction to Information Retrieval,IR,"Information retrieval studies how to index, search, and rank documents for user queries. Core topics include term weighting, inverted indexes, BM25, evaluation metrics, and relevance feedback."
D2,BM25 Ranking Explained,IR,"BM25 is a bag-of-words ranking function used by search engines to score documents against a query. It balances term frequency, inverse document frequency, and document length normalization."
D3,Semantic Search with Transformers,NLP,Semantic search uses dense embeddings from transformer models to capture meaning beyond exact keyword overlap. Sentence Transformers are widely used for query and document representation.
D4,Cosine Similarity for Vector Search,NLP,Vector retrieval often compares embeddings using cosine similarity. High cosine similarity indicates strong semantic alignment between a query vector and a document vector.
D5,Hybrid Retrieval Systems,IR,Hybrid retrieval combines lexical ranking such as BM25 with dense semantic retrieval. The goal is to improve search relevance by capturing both exact term matches and semantic meaning.
D6,Precision and Recall in Evaluation,ML,"Precision measures how many retrieved documents are relevant, while recall measures how many relevant documents were found. These metrics are standard in information retrieval evaluation."
D7,Mean Average Precision and MRR,ML,MAP summarizes ranking quality across queries using average precision. Mean reciprocal rank focuses on the position of the first relevant result.
D8,Streamlit for Search Applications,Apps,"Streamlit helps developers deploy interactive machine learning and search demos quickly. It supports widgets, tables, filters, and real-time query interfaces."
D9,Sentence Embeddings in Practice,NLP,"Sentence embeddings map text into dense vectors that preserve semantic similarity. They are useful for retrieval, clustering, recommendation, and duplicate detection."
D10,Inverted Index Fundamentals,IR,An inverted index maps terms to the documents that contain them. This data structure enables efficient keyword search at scale.


In [0]:
#create a cleaned Delta table

required_columns = {"doc_id", "title", "category", "text"}
missing_columns = required_columns - set(raw_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

clean_df = (
    raw_df
    .select(
        F.trim(F.col("doc_id").cast("string")).alias("doc_id"),
        F.trim(F.col("title").cast("string")).alias("title"),
        F.lower(F.trim(F.col("category").cast("string"))).alias("category"),
        F.regexp_replace(
            F.trim(F.col("text").cast("string")),
            r"\s+",
            " ",
        ).alias("text"),
    )
    .filter(F.col("doc_id").isNotNull())
    .filter(F.length("text") > 0)
    .dropDuplicates(["doc_id"])
    .withColumn("text_length", F.length("text"))
    .withColumn("word_count", F.size(F.split("text", r"\s+")))
    .withColumn("processed_at", F.current_timestamp())
)

clean_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.rag_corpus_clean")

print("Clean document count:", clean_df.count())
display(clean_df)

Clean document count: 12


doc_id,title,category,text,text_length,word_count,processed_at
D1,Introduction to Information Retrieval,ir,"Information retrieval studies how to index, search, and rank documents for user queries. Core topics include term weighting, inverted indexes, BM25, evaluation metrics, and relevance feedback.",192,26,2026-09-23T02:13:05.586Z
D2,BM25 Ranking Explained,ir,"BM25 is a bag-of-words ranking function used by search engines to score documents against a query. It balances term frequency, inverse document frequency, and document length normalization.",189,27,2026-09-23T02:13:05.586Z
D3,Semantic Search with Transformers,nlp,Semantic search uses dense embeddings from transformer models to capture meaning beyond exact keyword overlap. Sentence Transformers are widely used for query and document representation.,187,25,2026-09-23T02:13:05.586Z
D4,Cosine Similarity for Vector Search,nlp,Vector retrieval often compares embeddings using cosine similarity. High cosine similarity indicates strong semantic alignment between a query vector and a document vector.,172,23,2026-09-23T02:13:05.586Z
D5,Hybrid Retrieval Systems,ir,Hybrid retrieval combines lexical ranking such as BM25 with dense semantic retrieval. The goal is to improve search relevance by capturing both exact term matches and semantic meaning.,184,28,2026-09-23T02:13:05.586Z
D6,Precision and Recall in Evaluation,ml,"Precision measures how many retrieved documents are relevant, while recall measures how many relevant documents were found. These metrics are standard in information retrieval evaluation.",187,25,2026-09-23T02:13:05.586Z
D7,Mean Average Precision and MRR,ml,MAP summarizes ranking quality across queries using average precision. Mean reciprocal rank focuses on the position of the first relevant result.,145,21,2026-09-23T02:13:05.586Z
D8,Streamlit for Search Applications,apps,"Streamlit helps developers deploy interactive machine learning and search demos quickly. It supports widgets, tables, filters, and real-time query interfaces.",158,20,2026-09-23T02:13:05.586Z
D9,Sentence Embeddings in Practice,nlp,"Sentence embeddings map text into dense vectors that preserve semantic similarity. They are useful for retrieval, clustering, recommendation, and duplicate detection.",166,21,2026-09-23T02:13:05.586Z
D10,Inverted Index Fundamentals,ir,An inverted index maps terms to the documents that contain them. This data structure enables efficient keyword search at scale.,127,20,2026-09-23T02:13:05.586Z


In [0]:
#retrieval-sized chunks using distributed Spark transformations.

CHUNK_SIZE = 60
CHUNK_OVERLAP = 10
CHUNK_STEP = CHUNK_SIZE - CHUNK_OVERLAP

chunk_df = (
    clean_df
    .withColumn("tokens", F.split(F.col("text"), r"\s+"))
    .withColumn(
        "chunk_start",
        F.explode(
            F.sequence(
                F.lit(0),
                F.greatest(F.size("tokens") - 1, F.lit(0)),
                F.lit(CHUNK_STEP),
            )
        ),
    )
    .withColumn(
        "chunk_tokens",
        F.slice(
            F.col("tokens"),
            F.col("chunk_start") + 1,
            CHUNK_SIZE,
        ),
    )
    .withColumn("chunk_text", F.concat_ws(" ", F.col("chunk_tokens")))
    .withColumn(
        "chunk_index",
        (F.col("chunk_start") / F.lit(CHUNK_STEP)).cast("integer"),
    )
    .withColumn(
        "chunk_id",
        F.concat(
            F.col("doc_id"),
            F.lit("-chunk-"),
            F.lpad(F.col("chunk_index").cast("string"), 4, "0"),
        ),
    )
    .withColumn("chunk_word_count", F.size("chunk_tokens"))
    .filter(F.length("chunk_text") > 0)
    .select(
        "chunk_id",
        "doc_id",
        "title",
        "category",
        "chunk_index",
        "chunk_text",
        "chunk_word_count",
        "processed_at",
    )
)

chunk_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.rag_document_chunks")

print("Documents:", clean_df.count())
print("Chunks:", chunk_df.count())

display(chunk_df)

Documents: 12
Chunks: 12


chunk_id,doc_id,title,category,chunk_index,chunk_text,chunk_word_count,processed_at
D5-chunk-0000,D5,Hybrid Retrieval Systems,ir,0,Hybrid retrieval combines lexical ranking such as BM25 with dense semantic retrieval. The goal is to improve search relevance by capturing both exact term matches and semantic meaning.,28,2026-09-23T02:13:16.559Z
D2-chunk-0000,D2,BM25 Ranking Explained,ir,0,"BM25 is a bag-of-words ranking function used by search engines to score documents against a query. It balances term frequency, inverse document frequency, and document length normalization.",27,2026-09-23T02:13:16.559Z
D8-chunk-0000,D8,Streamlit for Search Applications,apps,0,"Streamlit helps developers deploy interactive machine learning and search demos quickly. It supports widgets, tables, filters, and real-time query interfaces.",20,2026-09-23T02:13:16.559Z
D11-chunk-0000,D11,Relevance Feedback Methods,ir,0,Relevance feedback improves retrieval by refining a query using judged relevant documents. Classic approaches include pseudo relevance feedback and relevance models.,21,2026-09-23T02:13:16.559Z
D4-chunk-0000,D4,Cosine Similarity for Vector Search,nlp,0,Vector retrieval often compares embeddings using cosine similarity. High cosine similarity indicates strong semantic alignment between a query vector and a document vector.,23,2026-09-23T02:13:16.559Z
D10-chunk-0000,D10,Inverted Index Fundamentals,ir,0,An inverted index maps terms to the documents that contain them. This data structure enables efficient keyword search at scale.,20,2026-09-23T02:13:16.559Z
D7-chunk-0000,D7,Mean Average Precision and MRR,ml,0,MAP summarizes ranking quality across queries using average precision. Mean reciprocal rank focuses on the position of the first relevant result.,21,2026-09-23T02:13:16.559Z
D9-chunk-0000,D9,Sentence Embeddings in Practice,nlp,0,"Sentence embeddings map text into dense vectors that preserve semantic similarity. They are useful for retrieval, clustering, recommendation, and duplicate detection.",21,2026-09-23T02:13:16.559Z
D6-chunk-0000,D6,Precision and Recall in Evaluation,ml,0,"Precision measures how many retrieved documents are relevant, while recall measures how many relevant documents were found. These metrics are standard in information retrieval evaluation.",25,2026-09-23T02:13:16.559Z
D3-chunk-0000,D3,Semantic Search with Transformers,nlp,0,Semantic search uses dense embeddings from transformer models to capture meaning beyond exact keyword overlap. Sentence Transformers are widely used for query and document representation.,25,2026-09-23T02:13:16.559Z


In [0]:
chunk_analytics = (
    chunk_df
    .groupBy("category")
    .agg(
        F.countDistinct("doc_id").alias("documents"),
        F.count("*").alias("chunks"),
        F.round(F.avg("chunk_word_count"), 2).alias("avg_chunk_words"),
        F.max("chunk_word_count").alias("max_chunk_words"),
    )
    .orderBy(F.desc("chunks"))
)

display(chunk_analytics)

category,documents,chunks,avg_chunk_words,max_chunk_words
ir,5,5,24.4,28
nlp,3,3,23.0,25
apps,2,2,20.5,21
ml,2,2,23.0,25


In [0]:
#to inspect Delta history

display(
    spark.sql(
        "DESCRIBE HISTORY workspace.default.rag_document_chunks"
    )
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-23T02:13:14.000Z,74700832567843,nikshitharapolu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2769211653063068),5436bdf4-bb07-4b6c-b368-79bfb04a73aa,0923-021022-9k2hd1sw-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 12, numOutputBytes -> 4648)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
embedding_test = spark.sql("""
    SELECT ai_query(
        "system.ai.gte-large-en",
        "Hybrid retrieval combines lexical and semantic search."
    ) AS embedding
""")

test_vector = embedding_test.first()["embedding"]

print("Embedding type:", type(test_vector))
print("Embedding dimensions:", len(test_vector))
print("First five values:", test_vector[:5])

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-7391065049740851>, line 8
      1 embedding_test = spark.sql("""
      2     SELECT ai_query(
      3         "system.ai.gte-large-en",
      4         "Hybrid retrieval combines lexical and semantic search."
      5     ) AS embedding
      6 """)
----> 8 test_vector = embedding_test.first()["embedding"]
     10 print("Embedding type:", type(test_vector))
     11 print("Embedding dimensions:", len(test_vector))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:576, in DataFrame.first(self)
    575 def first(self) -> Optional[Row]:
--> 576     return self.head()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:720, in DataFrame.head(self, n)
    718 def head(self, n: Optional[int] = None) -> Union[Optional[Row], List[Row]]:
    719     if n is None:

In [0]:
embedding_raw_df = spark.table(
    "workspace.default.rag_embeddings_raw"
)

embedding_df = (
    embedding_raw_df
    .withColumn(
        "embedding_vector",
        F.from_json(
            F.col("embedding"),
            "array<float>",
        ),
    )
    .drop("embedding")
    .filter(F.size("embedding_vector") == 384)
    .withColumn("indexed_at", F.current_timestamp())
)

embedding_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("workspace.default.rag_embeddings")

print("Embedding rows:", embedding_df.count())
print(
    "Embedding dimensions:",
    embedding_df.select(
        F.size("embedding_vector").alias("dimensions")
    ).first()["dimensions"],
)

display(
    embedding_df.select(
        "doc_id",
        "title",
        "category",
        "embedding_model",
        "embedding_dimensions",
    )
)

Embedding rows: 38
Embedding dimensions: 384


doc_id,title,category,embedding_model,embedding_dimensions
D1,Introduction to Information Retrieval,IR,sentence-transformers/all-MiniLM-L6-v2,384
D2,BM25 Ranking Explained,IR,sentence-transformers/all-MiniLM-L6-v2,384
D3,Semantic Search with Transformers,NLP,sentence-transformers/all-MiniLM-L6-v2,384
D4,Cosine Similarity for Vector Search,NLP,sentence-transformers/all-MiniLM-L6-v2,384
D5,Hybrid Retrieval Systems,IR,sentence-transformers/all-MiniLM-L6-v2,384
D6,Precision and Recall in Evaluation,ML,sentence-transformers/all-MiniLM-L6-v2,384
D7,Mean Average Precision and MRR,ML,sentence-transformers/all-MiniLM-L6-v2,384
D8,Streamlit for Search Applications,Apps,sentence-transformers/all-MiniLM-L6-v2,384
D9,Sentence Embeddings in Practice,NLP,sentence-transformers/all-MiniLM-L6-v2,384
D10,Inverted Index Fundamentals,IR,sentence-transformers/all-MiniLM-L6-v2,384


In [0]:
from pyspark.sql import Window

# Load document embeddings from the governed Delta table.
documents = spark.table(
    "workspace.default.rag_embeddings"
).select(
    "doc_id",
    "title",
    "category",
    "text",
    "embedding_vector",
)

# Parse the uploaded query vectors.
queries = (
    spark.table("workspace.default.rag_query_embeddings_raw")
    .withColumn(
        "query_vector",
        F.from_json(
            F.col("query_embedding"),
            "array<float>",
        ),
    )
    .select(
        F.col("query_id").cast("string").alias("query_id"),
        "query",
        "query_vector",
    )
    .filter(F.size("query_vector") == 384)
)

# Since MiniLM vectors were normalized during export, their dot product
# is equivalent to cosine similarity.
similarity_results = (
    documents.crossJoin(queries)
    .withColumn(
        "cosine_similarity",
        F.expr("""
            aggregate(
                zip_with(
                    embedding_vector,
                    query_vector,
                    (document_value, query_value) ->
                        cast(document_value as double) *
                        cast(query_value as double)
                ),
                cast(0.0 as double),
                (total, value) -> total + value
            )
        """),
    )
)

ranking_window = Window.partitionBy("query_id").orderBy(
    F.desc("cosine_similarity")
)

top_results = (
    similarity_results
    .withColumn(
        "rank",
        F.row_number().over(ranking_window),
    )
    .filter(F.col("rank") <= 5)
    .select(
        "query_id",
        "query",
        "rank",
        "doc_id",
        "title",
        "category",
        F.round("cosine_similarity", 6).alias(
            "cosine_similarity"
        ),
    )
)

top_results.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(
    "workspace.default.rag_vector_search_results"
)

print("Document vectors:", documents.count())
print("Query vectors:", queries.count())
print("Top-K results:", top_results.count())

display(
    top_results.orderBy("query_id", "rank")
)

Document vectors: 38
Query vectors: 5
Top-K results: 25


query_id,query,rank,doc_id,title,category,cosine_similarity
Q1,bm25 ranking for search engines,1,D2,BM25 Ranking Explained,IR,0.773372
Q1,bm25 ranking for search engines,2,D5,Hybrid Retrieval Systems,IR,0.62244
Q1,bm25 ranking for search engines,3,D1,Introduction to Information Retrieval,IR,0.616301
Q1,bm25 ranking for search engines,4,D7,Mean Average Precision and MRR,ML,0.500185
Q1,bm25 ranking for search engines,5,D10,Inverted Index Fundamentals,IR,0.446512
Q2,semantic search using sentence transformers,1,D3,Semantic Search with Transformers,NLP,0.776997
Q2,semantic search using sentence transformers,2,D5,Hybrid Retrieval Systems,IR,0.512389
Q2,semantic search using sentence transformers,3,D9,Sentence Embeddings in Practice,NLP,0.470058
Q2,semantic search using sentence transformers,4,D10,Inverted Index Fundamentals,IR,0.448233
Q2,semantic search using sentence transformers,5,D4,Cosine Similarity for Vector Search,NLP,0.420229


In [0]:
pipeline_summary = spark.createDataFrame(
    [
        ("raw_documents", spark.table(
            "workspace.default.rag_corpus_raw"
        ).count()),
        ("clean_documents", spark.table(
            "workspace.default.rag_corpus_clean"
        ).count()),
        ("document_chunks", spark.table(
            "workspace.default.rag_document_chunks"
        ).count()),
        ("embedding_vectors", documents.count()),
        ("query_vectors", queries.count()),
        ("top_k_results", top_results.count()),
    ],
    ["pipeline_stage", "row_count"],
)

pipeline_summary.write.format("delta").mode(
    "overwrite"
).saveAsTable(
    "workspace.default.rag_pipeline_summary"
)

display(pipeline_summary)

pipeline_stage,row_count
raw_documents,12
clean_documents,12
document_chunks,12
embedding_vectors,38
query_vectors,5
top_k_results,25
